# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)
This notebook simulates streaming ingestion of synthetic FHIR Observation & Patient streams via PySpark memory streams into Bronze.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, expr
import time

spark = SparkSession.builder \
    .appName("Day02_Streaming_Simulation") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

print("Spark session configured for streaming.")

In [ ]:
# Generate synthetic streaming data using rate source
streaming_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

# Transform into our bronze format
simulated_fhir_df = streaming_df \
    .withColumn("payload_id", expr("uuid()")) \
    .withColumn("source_system_id", expr("'SIMULATOR_FHIR'")) \
    .withColumn("raw_payload_json", expr("concat('{\"resourceType\": \"Patient\", \"id\": \"', cast(value as string), '\"}')")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "raw_payload_json", "ingestion_timestamp")

# Write stream into the Iceberg table created in Day 1
# We'll use memory format for demonstration purposes if Iceberg is not fully setup locally to receive streams in Colab
query = simulated_fhir_df.writeStream \
    .format("memory") \
    .queryName("bronze_stream_sink") \
    .outputMode("append") \
    .start()

print("Started simulated streaming into memory sink.")
time.sleep(10)
query.stop()

print("Showing results from memory sink:")
spark.sql("SELECT * FROM bronze_stream_sink").show(truncate=False)